# WAXAL-NET fine-tune: akan × whisper_tiny

Trains on WAXAL akan, evaluates on the WAXAL test split.

Needs GPU T4 x2, internet, and the private `waxal-hf-token` dataset (an HF write token) so checkpoints survive the 12 h session limit. If a run stops at the deadline, run it again to resume.

In [ ]:
LANGUAGE = "akan"
MODEL = "whisper_tiny"
FLEURS = False
TRAIN_HOURS = 10.5

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

SESSION_START = time.time()


def find_repo():
    for depth in range(1, 5):
        for hit in Path("/kaggle/input").glob("*/" * depth + "src/train.py"):
            return hit.parent.parent
    raise FileNotFoundError("Code not found: attach the waxal-asr-code dataset (Add Input).")


REPO = find_repo()
PY = "/tmp/waxal-venv/bin/python"
os.environ.update(HF_HOME="/tmp/hf_cache", HF_DATASETS_DISABLE_PROGRESS_BARS="1", WANDB_DISABLED="true",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
                  # One GPU: Trainer's DataParallel over both T4s made MMS-300M steps ~6.5x
                  # slower (kaggle/bench_mms.py). Accumulation doubles to keep batch 32.
                  CUDA_VISIBLE_DEVICES="0")
TOKEN_FILES = sorted(Path("/kaggle/input").glob("**/hf_token.txt"))
if TOKEN_FILES:
    os.environ["HF_TOKEN"] = TOKEN_FILES[0].read_text().strip()
else:
    print("No hf_token.txt input (waxal-hf-token dataset); checkpoints will not survive this session.")


LOG_DIR = Path("/kaggle/working/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run(*command, check=True):
    """Run a command from the repo root, streaming its output to the notebook and to
    /kaggle/working/logs (kept in the job output; the API's notebook log can be empty)."""
    name = "_".join(Path(str(part)).stem for part in command[1:4])
    log_path = LOG_DIR / f"{time.strftime('%H%M%S')}_{name}.log"
    process = subprocess.Popen(list(command), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    with open(log_path, "w", encoding="utf-8") as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    if process.wait() and check:
        raise RuntimeError(f"{' '.join(map(str, command))} exited with {process.returncode}")
    return process.returncode


print("Code:", REPO)

In [ ]:
run("bash", f"{REPO}/kaggle/setup_env.sh")

In [ ]:
RUN = f"finetune_{LANGUAGE}_{MODEL}"
CONFIG = f"configs/experiment/{RUN}.yaml"
CHECKPOINT = f"/kaggle/working/checkpoints/{RUN}/best_model"
HUB = ["--hub-repo", f"waxal-net-{RUN}"] if os.environ.get("HF_TOKEN") else []
run(PY, "-m", "src.train", "--config", CONFIG, "--deadline", str(SESSION_START + TRAIN_HOURS * 3600), *HUB)

In [ ]:
if Path(CHECKPOINT, "config.json").exists():
    run(PY, "-m", "src.evaluate", "--config", CONFIG, "--checkpoint", CHECKPOINT)
    if MODEL.startswith("whisper"):
        # Also score clips over 30 s in 30 s chunks; the paper doesn't say which it did.
        run(PY, "-m", "src.evaluate", "--config", CONFIG, "--checkpoint", CHECKPOINT, "--long-audio", "chunk30",
            "--output-dir", f"/kaggle/working/outputs/{RUN}_chunk30")
    if FLEURS:
        run(PY, "-m", "src.cross_domain_eval", "--config", f"configs/experiment/cross_domain_{LANGUAGE}_{MODEL}.yaml",
            "--checkpoint", CHECKPOINT)
else:
    print("Training stopped at the session deadline. Run this notebook again to resume from the last checkpoint.")

In [ ]:
import json, shutil

for name in (f"finetune_{LANGUAGE}_{MODEL}", f"finetune_{LANGUAGE}_{MODEL}_chunk30", f"cross_domain_{LANGUAGE}_{MODEL}"):
    path = Path(f"/kaggle/working/outputs/{name}/metrics.json")
    if path.exists():
        metrics = json.loads(path.read_text())
        print(f"{name}: WER {metrics['wer_percent']:.1f}% | CER {metrics['cer_percent']:.1f}%")

# Keep the saved notebook output small. With HF_TOKEN every checkpoint and the
# final model are already on the Hub; without it only best_model is kept.
checkpoints = Path(f"/kaggle/working/checkpoints/{RUN}")
if os.environ.get("HF_TOKEN"):
    shutil.rmtree(checkpoints, ignore_errors=True)
else:
    for intermediate in checkpoints.glob("checkpoint-*"):
        shutil.rmtree(intermediate)